# CASML B0 — generate

Chỉ chạy bước generate. Chỉnh đường dẫn artifact đã có trong cell cấu hình.

Generation chỉ đọc cache retrieval; không import FAISS/SentenceTransformer, không cần corpus/index.

In [ ]:
from pathlib import Path
import subprocess, sys

# Notebook tự clone mã nguồn khi chưa có project. Bật Internet trên Kaggle.
REPO_URL = "https://github.com/lighth-gh/casml-rag.git"
REPO_REF = "main"
ROOT_OVERRIDE = None  # Hoặc đường dẫn tới một bản project đã có.
UPDATE_REPO = False   # Đổi thành True để git pull --ff-only khi chạy lại cell.

def is_project(path):
    return (path / "pyproject.toml").is_file() and (path / "casml_b0/cli.py").is_file()

if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    local = [Path.cwd(), *Path.cwd().parents]
    candidates = list(dict.fromkeys(p.resolve() for p in local if is_project(p)))
    if len(candidates) > 1:
        raise RuntimeError(f"Tìm thấy nhiều project: {candidates}. Hãy đặt ROOT_OVERRIDE.")
    if candidates:
        ROOT = candidates[0]
    else:
        base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
        ROOT = (base / "casml-rag").resolve()
        if ROOT.exists() and not is_project(ROOT):
            raise RuntimeError(f"{ROOT} đã tồn tại nhưng không phải project CASML; hãy đổi ROOT_OVERRIDE hoặc tên thư mục.")
        if not ROOT.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)],
                check=True,
            )

if not is_project(ROOT):
    raise FileNotFoundError(f"Không tìm thấy project CASML tại {ROOT}")
if UPDATE_REPO:
    if not (ROOT / ".git").is_dir():
        raise RuntimeError("UPDATE_REPO chỉ dùng được với thư mục được git clone.")
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", REPO_REF], check=True)

if Path("/kaggle/working").exists():
    # Artifact nằm ngoài repo để git pull không đụng vào kết quả đã chạy.
    WORK = Path("/kaggle/working/casml_b0_work")
else:
    WORK = ROOT
WORK.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Artifacts:", WORK)

def stage(*args):
    subprocess.run([sys.executable, "-m", "casml_b0", *map(str, args)], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-generation.txt")], check=True)


In [ ]:
# SỬA đường dẫn cho đúng dữ liệu của bạn. Không dùng PDF overview làm sách.
DATA = Path("/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)") if Path("/kaggle/input").exists() else ROOT / "data"
PDF = DATA / "book.pdf"
QUERIES = DATA / "queries.json"
SAMPLE = None  # Đặt Path đến sample_submission.csv chính thức nếu có.
PAGE_MAP = None  # Đặt Path đến page_map_override.csv sau khi kiểm tra số trang/mục.
CORPUS = WORK / "artifacts/corpus_v1"
INDEX = WORK / "artifacts/index_v1"
RETRIEVAL = WORK / "artifacts/retrieval_v1"
RUN = WORK / "runs/b0_g01"
OUTPUT = WORK / "outputs/b0_g01"
GEN_CONFIG = ROOT / "configs/generate.yaml"
# Khi đổi prompt/model/config, dùng RUN và OUTPUT mới.


In [ ]:
# Nếu cache nằm ở Input, sửa RETRIEVAL tại đây, ví dụ:
# RETRIEVAL = Path("/kaggle/input/your-retrieval-cache/retrieval_v1")
# Chỉ cần manifest.json + retrieval.jsonl. Không cần PDF/index/embedding.
stage("generate", "--retrieval", RETRIEVAL, "--config", GEN_CONFIG, "--out", RUN)
print((RUN / "report.json").read_text())
